In [1]:
import pandas as pd
import requests

url = "https://servicodados.ibge.gov.br/api/v1/localidades/municipios"
response = requests.get(url)

if response.status_code == 200:
    dados_json = response.json()
    
    #Transforma em dataframe
    df_bruto = pd.DataFrame(dados_json)
    
    display(df_bruto.head())
else:
    print(f"Falhou: {response.status_code}")

,id,nome,microrregiao,regiao-imediata
0,1100015,Alta Floresta D'Oeste,"{'id': 11006, 'nome': 'Cacoal', 'mesorregiao':...","{'id': 110005, 'nome': 'Cacoal', 'regiao-inter..."
1,1100023,Ariquemes,"{'id': 11003, 'nome': 'Ariquemes', 'mesorregia...","{'id': 110002, 'nome': 'Ariquemes', 'regiao-in..."
2,1100031,Cabixi,"{'id': 11008, 'nome': 'Colorado do Oeste', 'me...","{'id': 110006, 'nome': 'Vilhena', 'regiao-inte..."
3,1100049,Cacoal,"{'id': 11006, 'nome': 'Cacoal', 'mesorregiao':...","{'id': 110005, 'nome': 'Cacoal', 'regiao-inter..."
4,1100056,Cerejeiras,"{'id': 11008, 'nome': 'Colorado do Oeste', 'me...","{'id': 110006, 'nome': 'Vilhena', 'regiao-inte..."


In [3]:
df_limpo = pd.DataFrame()

# cod_ibge e Nome da cidade
df_limpo['cod_ibge'] = df_bruto['id']
df_limpo['nome_municipio'] = df_bruto['nome']

# Uf
df_limpo['uf'] = df_bruto['microrregiao'].apply(lambda x: x['mesorregiao']['UF']['sigla'] if x and 'mesorregiao' in x and x['mesorregiao'] and 'UF' in x['mesorregiao'] and x['mesorregiao']['UF'] else None)

# estado
df_limpo['estado'] = df_bruto['microrregiao'].apply(lambda x: x['mesorregiao']['UF']['nome'] if x and 'mesorregiao' in x and x['mesorregiao'] and 'UF' in x['mesorregiao'] and x['mesorregiao']['UF'] else None)

# regiao
df_limpo['regiao'] = df_bruto['microrregiao'].apply(lambda x: x['mesorregiao']['UF']['regiao']['nome'] if x and 'mesorregiao' in x and x['mesorregiao'] and 'UF' in x['mesorregiao'] and x['mesorregiao']['UF'] and 'regiao' in x['mesorregiao']['UF'] and x['mesorregiao']['UF']['regiao'] else None)

df_limpo = df_limpo.drop_duplicates(subset=['cod_ibge'])

display(df_limpo.head(10))

,cod_ibge,nome_municipio,uf,estado,regiao
0,1100015,Alta Floresta D'Oeste,RO,Rondônia,Norte
1,1100023,Ariquemes,RO,Rondônia,Norte
2,1100031,Cabixi,RO,Rondônia,Norte
3,1100049,Cacoal,RO,Rondônia,Norte
4,1100056,Cerejeiras,RO,Rondônia,Norte
5,1100064,Colorado do Oeste,RO,Rondônia,Norte
6,1100072,Corumbiara,RO,Rondônia,Norte
7,1100080,Costa Marques,RO,Rondônia,Norte
8,1100098,Espigão D'Oeste,RO,Rondônia,Norte
9,1100106,Guajará-Mirim,RO,Rondônia,Norte


In [4]:
#Verificar se ha dados nulos.
df_bruto.count().isnull()

id                 False
nome               False
microrregiao       False
regiao-imediata    False
dtype: bool

In [4]:
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

usuario = 'root'
senha = '4bobr1nh4d0sUd@0'
host = '127.0.0.1'
porta = 3306
banco = 'tcc_epidemiologia'

# Conexao do banco
url_conexao = URL.create(
    "mysql+pymysql",
    username=usuario,
    password=senha,
    host=host,
    port=porta,
    database=banco,
 )
engine = create_engine(url_conexao)

print("Enviando municipios para o banco local...")

try:
    # Testa a conexao antes de gravar
    with engine.connect() as conn:
        conn.execute(text("SELECT 1"))

    df_limpo.to_sql('municipios', con=engine, if_exists='replace', index=False)
    print("municipios carregados com sucesso!")
except Exception as e:
    print(f"Erro na carga: {e}")

Enviando municipios para o banco local...
municipios carregados com sucesso!
